**Name:** RATUL SIKDER
**Roll:** 2506102
**Course:** MITE 431 - Big Data Analytics

---

# Part A: Supervised Learning — Titanic Survival Prediction

Predicts whether a passenger survived the Titanic disaster using **PySpark MLlib**
(Logistic Regression). Dataset: `dataset/Titanic-Dataset.csv` (Kaggle).

## Task 1 — Create a SparkSession and load the CSV

In [19]:
import warnings
warnings.filterwarnings('ignore')
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, mean
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator,
)

master_url = os.environ.get("SPARK_MASTER_URL", "local[*]")

spark = (
    SparkSession.builder
    .appName("TitanicSurvivalPrediction")
    .master(master_url)
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| master:", master_url)

Spark 4.1.2 | master: spark://spark-master:7077


In [20]:
df = spark.read.csv("dataset/Titanic-Dataset.csv", header=True, inferSchema=True)
print("Rows:", df.count())

Rows: 891


## Task 2 — Explore the dataset: schema and basic statistics

In [21]:
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [22]:
df.summary("count", "min", "max", "50%").show()

+-------+-----------+--------+------+--------------------+------+----+-----+-----+---------+--------+-----+--------+
|summary|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|   Ticket|    Fare|Cabin|Embarked|
+-------+-----------+--------+------+--------------------+------+----+-----+-----+---------+--------+-----+--------+
|  count|        891|     891|   891|                 891|   891| 714|  891|  891|      891|     891|  204|     889|
|    min|          1|       0|     1|"Andersson, Mr. A...|female|0.42|    0|    0|   110152|     0.0|  A10|       C|
|    max|        891|       1|     3|van Melkebeke, Mr...|  male|80.0|    8|    6|WE/P 5735|512.3292|    T|       S|
|    50%|        446|       0|     3|                NULL|  NULL|28.0|    0|    0| 236171.0| 14.4542| NULL|    NULL|
+-------+-----------+--------+------+--------------------+------+----+-----+-----+---------+--------+-----+--------+



In [23]:
df.describe("Survived", "Pclass", "Age", "SibSp", "Parch", "Fare").show()

+-------+-------------------+------------------+------------------+------------------+-------------------+-----------------+
|summary|           Survived|            Pclass|               Age|             SibSp|              Parch|             Fare|
+-------+-------------------+------------------+------------------+------------------+-------------------+-----------------+
|  count|                891|               891|               714|               891|                891|              891|
|   mean| 0.3838383838383838| 2.308641975308642| 29.69911764705882|0.5230078563411896|0.38159371492704824| 32.2042079685746|
| stddev|0.48659245426485753|0.8360712409770491|14.526497332334035|1.1027434322934315| 0.8060572211299488|49.69342859718089|
|    min|                  0|                 1|              0.42|                 0|                  0|              0.0|
|    max|                  1|                 3|              80.0|                 8|                  6|         512.3292|


## Task 3 — Identify and handle missing values

In [24]:
df.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
).show()

+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+
|PassengerId|Survived|Pclass|Name|Sex|Age|SibSp|Parch|Ticket|Fare|Cabin|Embarked|
+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+
|          0|       0|     0|   0|  0|177|    0|    0|     0|   0|  687|       2|
+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+



In [25]:
# Impute Age with the mean, drop the sparse Cabin column, fill Embarked with the mode
mean_age = df.select(mean(col("Age"))).first()[0]
df = df.drop("Cabin")
df = df.fillna({"Age": mean_age, "Embarked": "S"})
print(f"Imputed missing Age with mean = {mean_age:.2f}")

Imputed missing Age with mean = 29.70


## Task 4 — Select input features

In [26]:
feature_cols = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
label_col = "Survived"
df = df.select(feature_cols + [label_col])
df.show(5)

+------+------+----+-----+-----+-------+--------+--------+
|Pclass|   Sex| Age|SibSp|Parch|   Fare|Embarked|Survived|
+------+------+----+-----+-----+-------+--------+--------+
|     3|  male|22.0|    1|    0|   7.25|       S|       0|
|     1|female|38.0|    1|    0|71.2833|       C|       1|
|     3|female|26.0|    0|    0|  7.925|       S|       1|
|     1|female|35.0|    1|    0|   53.1|       S|       1|
|     3|  male|35.0|    0|    0|   8.05|       S|       0|
+------+------+----+-----+-----+-------+--------+--------+
only showing top 5 rows


## Task 5 — Encode categorical attributes 

In [27]:
sex_indexer = StringIndexer(inputCol="Sex", outputCol="SexIndex")
embarked_indexer = StringIndexer(inputCol="Embarked", outputCol="EmbarkedIndex")
encoder = OneHotEncoder(
    inputCols=["SexIndex", "EmbarkedIndex"],
    outputCols=["SexVec", "EmbarkedVec"],
)

## Task 6 — Combine features into a single vector

In [28]:
assembler = VectorAssembler(
    inputCols=["Pclass", "SexVec", "Age", "SibSp", "Parch", "Fare", "EmbarkedVec"],
    outputCol="features",
)

## Task 7 — Train/test split (80/20)

In [29]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print(f"Training rows: {train_df.count()}, Test rows: {test_df.count()}")

Training rows: 746, Test rows: 145


## Task 8 — Train a Logistic Regression classifier (Pipeline)

In [30]:
lr = LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=100)
pipeline = Pipeline(stages=[sex_indexer, embarked_indexer, encoder, assembler, lr])
model = pipeline.fit(train_df)

## Task 9 — Generate predictions on the test set

In [31]:
predictions = model.transform(test_df)

## Task 10 — Evaluate: Accuracy, Precision, Recall, F1, AUC

In [32]:
metrics = {}
for metric in ["accuracy", "weightedPrecision", "weightedRecall", "f1"]:
    evaluator = MulticlassClassificationEvaluator(
        labelCol=label_col, predictionCol="prediction", metricName=metric
    )
    metrics[metric] = evaluator.evaluate(predictions)

auc = BinaryClassificationEvaluator(
    labelCol=label_col, metricName="areaUnderROC"
).evaluate(predictions)

print(f"Accuracy  : {metrics['accuracy']:.4f}")
print(f"Precision : {metrics['weightedPrecision']:.4f} (weighted)")
print(f"Recall    : {metrics['weightedRecall']:.4f} (weighted)")
print(f"F1-score  : {metrics['f1']:.4f}")
print(f"AUC (ROC) : {auc:.4f}")

Accuracy  : 0.8276
Precision : 0.8261 (weighted)
Recall    : 0.8276 (weighted)
F1-score  : 0.8265
AUC (ROC) : 0.8941


## Task 11 — Sample predictions

In [33]:
predictions.select(
    "Pclass", "Sex", "Age", "Fare", "Survived", "prediction", "probability"
).show(20, truncate=False)

+------+------+-----------------+--------+--------+----------+-----------------------------------------+
|Pclass|Sex   |Age              |Fare    |Survived|prediction|probability                              |
+------+------+-----------------+--------+--------+----------+-----------------------------------------+
|1     |female|15.0             |211.3375|1       |1.0       |[0.03295170193328516,0.9670482980667149] |
|1     |female|17.0             |57.0    |1       |1.0       |[0.06308658658314004,0.93691341341686]   |
|1     |female|18.0             |79.65   |1       |1.0       |[0.0516366343264713,0.9483633656735287]  |
|1     |female|19.0             |91.0792 |1       |1.0       |[0.0420253847818437,0.9579746152181563]  |
|1     |female|22.0             |66.6    |1       |1.0       |[0.07328111444908658,0.9267188855509134] |
|1     |female|24.0             |69.3    |1       |1.0       |[0.03695493126715957,0.9630450687328405] |
|1     |female|29.0             |211.3375|1       |1.0 

### Discussion

The Logistic Regression model reaches **~82.8% accuracy** (F1 ≈ 0.83,
AUC ≈ 0.89) on the held-out test set. Preprocessing decisions: missing `Age`
values were imputed with the mean (29.70), the very sparse `Cabin` column was
dropped, and the two missing `Embarked` values were filled with the most
frequent port ("S"). Non-predictive identifiers (`PassengerId`, `Name`,
`Ticket`) were excluded. The model clearly learned the dominant survival
signals — sex, passenger class and fare — as first-class female passengers
receive survival probabilities above 90%.

In [34]:
spark.stop()